# Notebook 5: RUL Prediction ─ Remaining Useful Life
## ESP Predictive Maintenance

**Task**: Given a window of sensor readings, predict how many timesteps remain before the next failure event.

**Why it matters**: An anomaly alarm tells you *something is wrong now*. RUL tells you *how much time you have*. Together they enable:
- Proactive workover scheduling (before failure, not after)
- Inventory pre-positioning (order parts while pump still runs)
- Production deferral planning (when to shut in voluntarily)

**Datasets**:
1. **NASA CMAPSS** ─ gold standard RUL benchmark (turbofan engines, same methodology)
2. **Synthetic ESP** ─ with ground-truth RUL labels from generator
3. **Pump Sensor CSV** ─ real Kaggle data from Google Drive

**Loss**: Asymmetric weighted MSE ─ late predictions (over-predicting RUL) penalised 2x vs early predictions.


In [ ]:
# -- Cell 1: Environment setup -------------------------------------------------
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the full repo (Colab only copies this notebook by default)
    if not os.path.exists('src'):
        !git clone https://github.com/WickDager/esp-predictive-maintenance.git
        %cd esp-predictive-maintenance

    !pip install -q torch torchvision tqdm scikit-learn

    # Try Google Drive, fall back to local if mount fails
    USE_DRIVE = False
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        SAVE_DIR = '/content/drive/MyDrive/esp_pm/checkpoints'
        os.makedirs(SAVE_DIR, exist_ok=True)
        USE_DRIVE = True
        print('Google Drive mounted successfully.')
    except Exception as e:
        print(f'Drive mount failed ({e}). Using local storage instead.')
        SAVE_DIR = '/content/checkpoints'

    if not USE_DRIVE:
        SAVE_DIR = '/content/checkpoints'
else:
    SAVE_DIR = 'checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)
print(f'Checkpoints will be saved to: {SAVE_DIR}')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# -- Cell 2: Imports -------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath("."))
from src.data.loader import TimeSeriesDataset, make_dataloaders
from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
from src.data.loader import _sliding_window, _compute_rul, _split_and_scale, load_cmapss
from src.models.rul_predictor import RULPredictor, AsymmetricRULLoss, train_rul_epoch, evaluate_rul
from src.utils.metrics import rul_metrics
from src.utils.visualization import plot_rul_prediction
from src.training.trainer import EarlyStopping

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3})
print('Imports OK.')

In [ ]:
# -- Cell 3: Config --------------------------------------------------------
CONFIG = {
    'dataset':        'synthetic',   # 'synthetic' | 'pump_sensor' | 'cmapss'
    'cmapss_subset':  'FD001',
    'window_size':    50,
    'step_size':       5,            # larger step to avoid overfitting
    'clip_rul':       200,           # cap RUL (piecewise linear target)
    'val_split':     0.15,
    'test_split':    0.15,

    'hidden_size':   128,
    'num_layers':      3,
    'dropout':       0.3,
    'alpha':         2.0,            # asymmetric loss penalty for over-prediction

    'batch_size':    128,
    'num_epochs':    100,
    'lr':           1e-3,
    'weight_decay': 1e-4,
    'patience':       20,
    'gradient_clip': 1.0,
    'mc_samples':     50,
}
print('CONFIG ready.')

In [ ]:
# -- Cell 4: Load data -----------------------------------------------------
# CHOOSE YOUR DATA SOURCE in CONFIG:
#   'synthetic'   - generated on the fly
#   'pump_sensor' - load CSV from Google Drive
#   'cmapss'      - NASA turbofan dataset

if CONFIG['dataset'] == 'pump_sensor':
    DRIVE_CSV_PATH = '/content/drive/MyDrive/pump_sensor.csv'
    print(f'Loading real pump sensor data from {DRIVE_CSV_PATH}...')

    raw_df = pd.read_csv(DRIVE_CSV_PATH, parse_dates=['timestamp'])
    raw_df = raw_df.sort_values('timestamp').reset_index(drop=True)
    SENSOR_COLS = sorted([c for c in raw_df.columns if c.startswith('sensor_')])
    raw_df[SENSOR_COLS] = raw_df[SENSOR_COLS].ffill().bfill()

    status_map = {'NORMAL': 0, 'RECOVERING': 0, 'BROKEN': 1}
    raw_df['failure'] = raw_df['machine_status'].map(status_map).fillna(0).astype(int)
    print(f'Found {len(SENSOR_COLS)} sensors, failure rate: {raw_df["failure"].mean()*100:.2f}%')

    X_raw = raw_df[SENSOR_COLS].values.astype(np.float32)
    y_raw = raw_df['failure'].values.astype(np.float32)
    rul_raw = _compute_rul(y_raw)
    rul_raw = np.clip(rul_raw, 0, CONFIG['clip_rul']).astype(np.float32)
    rul_raw[rul_raw == CONFIG['clip_rul']] = -1

    X_w, y_w, rul_w = _sliding_window(
        X_raw, y_raw, rul_raw,
        CONFIG['window_size'], CONFIG['step_size']
    )
    data = _split_and_scale(X_w, y_w, rul_w, SENSOR_COLS,
                             CONFIG['val_split'], CONFIG['test_split'], 42)

elif CONFIG['dataset'] == 'synthetic':
    print('Generating synthetic data...')
    raw_df = generate_esp_dataset(n_wells=50, timesteps_per_well=4000, random_seed=42)
    SENSOR_COLS = SYNTHETIC_SENSOR_COLS
    raw_df['failure'] = (raw_df['machine_status'] == 'BROKEN').astype(int)
    raw_df[SENSOR_COLS] = raw_df[SENSOR_COLS].ffill().fillna(0)

    X_raw = raw_df[SENSOR_COLS].values.astype(np.float32)
    y_raw = raw_df['failure'].values.astype(np.float32)
    rul_raw = _compute_rul(y_raw)
    rul_raw = np.clip(rul_raw, 0, CONFIG['clip_rul']).astype(np.float32)
    rul_raw[rul_raw == CONFIG['clip_rul']] = -1

    X_w, y_w, rul_w = _sliding_window(
        X_raw, y_raw, rul_raw,
        CONFIG['window_size'], CONFIG['step_size']
    )
    data = _split_and_scale(X_w, y_w, rul_w, SENSOR_COLS,
                             CONFIG['val_split'], CONFIG['test_split'], 42)

elif CONFIG['dataset'] == 'cmapss':
    from src.data.loader import load_cmapss
    data = load_cmapss(
        data_dir='../data/raw/cmapss',
        subset=CONFIG['cmapss_subset'],
        window_size=CONFIG['window_size'],
        step_size=CONFIG['step_size'],
        clip_rul=130,
        val_split=CONFIG['val_split'],
    )
    SENSOR_COLS = data['feature_names']
    raw_df = None

n_features = data['X_train'].shape[2]
print(f'Train: {data["X_train"].shape}, Val: {data["X_val"].shape}, Test: {data["X_test"].shape}')
print(f'RUL range (train): {data["rul_train"][data["rul_train"]>=0].min():.0f} - {data["rul_train"].max():.0f}')

In [ ]:
# -- Cell 5: Visualise RUL distribution ------------------------------------
valid_rul = data['rul_train'][data['rul_train'] >= 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(valid_rul, bins=50, color='#378ADD', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Remaining Useful Life (timesteps)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('RUL Distribution (Training Set)', fontsize=12)

if raw_df is not None and 'well_id' in raw_df.columns:
    well = raw_df[raw_df['well_id'] == 0].reset_index(drop=True)
    axes[1].plot(well.index, np.clip(well['rul'], 0, CONFIG['clip_rul']),
                 color='#E24B4A', linewidth=1.5)
    axes[1].set_xlabel('Timestep', fontsize=11)
    axes[1].set_ylabel('RUL', fontsize=11)
    axes[1].set_title('RUL Countdown - One Well', fontsize=12)
else:
    axes[1].hist(data['rul_test'][data['rul_test'] >= 0], bins=30,
                 color='#E24B4A', alpha=0.8, edgecolor='white')
    axes[1].set_title('RUL Distribution (Test Set)', fontsize=12)

plt.tight_layout()
plt.savefig('results/05_rul_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- Cell 6: DataLoaders ---------------------------------------------------
train_ds = TimeSeriesDataset(data['X_train'], data['y_train'], data['rul_train'])
val_ds   = TimeSeriesDataset(data['X_val'],   data['y_val'],   data['rul_val'])
test_ds  = TimeSeriesDataset(data['X_test'],  data['y_test'],  data['rul_test'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
print(f'Train batches: {len(train_loader)}')

In [ ]:
# -- Cell 7: Build model ---------------------------------------------------
model = RULPredictor(
    input_size=n_features,
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    output_range=(0, CONFIG['clip_rul']),
).to(DEVICE)

criterion = AsymmetricRULLoss(alpha=CONFIG['alpha'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                               weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-5
)
early_stopper = EarlyStopping(patience=CONFIG['patience'])

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'RULPredictor | Parameters: {n_params:,}')

In [ ]:
# -- Cell 8: Training loop -------------------------------------------------
best_val_rmse = float('inf')
history = {'train_loss': [], 'val_rmse': [], 'val_mae': []}
best_ckpt = os.path.join(SAVE_DIR, 'best_rul_model.pt')

pbar = tqdm(range(1, CONFIG['num_epochs'] + 1), desc='Training RUL')
for epoch in pbar:
    train_loss = train_rul_epoch(model, train_loader, optimizer, criterion,
                                  DEVICE, CONFIG['gradient_clip'])
    val_metrics = evaluate_rul(model, val_loader, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_rmse'].append(val_metrics['rmse'])
    history['val_mae'].append(val_metrics['mae'])

    pbar.set_postfix({'loss': f"{train_loss:.3f}",
                       'RMSE': f"{val_metrics['rmse']:.1f}",
                       'MAE':  f"{val_metrics['mae']:.1f}"})

    if val_metrics['rmse'] < best_val_rmse:
        best_val_rmse = val_metrics['rmse']
        torch.save(model.state_dict(), best_ckpt)

    if early_stopper(val_metrics['rmse']):
        print(f'Early stopping at epoch {epoch}')
        break

# Load best
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
print(f'Best val RMSE: {best_val_rmse:.2f}')

In [ ]:
# -- Cell 9: Training curves -----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], label='Train loss', color='#378ADD')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Asymmetric MSE Loss')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[1].plot(history['val_rmse'], label='Val RMSE', color='#E24B4A')
axes[1].plot(history['val_mae'],  label='Val MAE',  color='#1D9E75')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Timesteps')
axes[1].set_title('Validation Metrics'); axes[1].legend()
plt.tight_layout()
plt.savefig('results/05_rul_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- Cell 10: Test evaluation ----------------------------------------------
test_metrics = evaluate_rul(model, test_loader, DEVICE)
print('='*45)
print('RUL PREDICTOR - TEST RESULTS')
print('='*45)
for k, v in test_metrics.items():
    print(f'  {k:20s}: {v:.4f}')
print('='*45)

In [ ]:
# -- Cell 11: MC Dropout - RUL with uncertainty ----------------------------
X_sample = torch.from_numpy(data['X_test'][:500]).float().to(DEVICE)
rul_mc = model.predict_with_uncertainty(X_sample, n_samples=CONFIG['mc_samples'])
rul_true = data['rul_test'][:500]
valid = rul_true >= 0

fig = plot_rul_prediction(
    y_true=rul_true[valid],
    y_pred=rul_mc['mean'][valid],
    ci_low=rul_mc['ci_low'][valid],
    ci_high=rul_mc['ci_high'][valid],
    title='Bi-LSTM RUL - Prediction with 90% CI (MC Dropout)',
    save_path='results/05_rul_mc_prediction.png',
)
plt.show()

print(f'Mean prediction uncertainty (sigma): {rul_mc["std"][valid].mean():.2f} timesteps')
print('Operator alert rule: flag if rul_mean < 48h AND rul_ci_low < 24h')

In [ ]:
# -- Cell 12: Save model ---------------------------------------------------
model.save_pretrained(SAVE_DIR)
full_cfg = {**CONFIG, 'feature_names': SENSOR_COLS, 'test_rmse': test_metrics['rmse']}
with open(os.path.join(SAVE_DIR, 'training_config.json'), 'w') as f:
    json.dump(full_cfg, f, indent=2)
print(f'RUL model saved to {SAVE_DIR}')
print('Next: Open notebooks/06_Survival_Analysis.ipynb')

## Benchmark: Synthetic vs CMAPSS

Run this cell to train and evaluate on **both** datasets, then get a side-by-side comparison table.

**Note:** This will re-train both models (~15-30 min total). Requires CMAPSS data to be downloaded first.

In [ ]:
# -- Cell 13: Benchmark Mode -----------------------------------------------
from collections import OrderedDict

def train_and_evaluate_on_dataset(
    dataset_name, data, sensor_cols, config, device, save_prefix
):
    """Train RUL model on a dataset and return test metrics."""
    print(f"\n{'='*60}")
    print(f"Training on: {dataset_name}")
    print(f"{'='*60}")

    n_feat = data['X_train'].shape[2]
    rul_valid = data['rul_train'][data['rul_train'] >= 0]
    clip_val = config['clip_rul']
    output_range = (0.0, float(np.percentile(rul_valid, 99))) if len(rul_valid) > 0 else None

    model = RULPredictor(
        input_size=n_feat, hidden_size=config['hidden_size'],
        num_layers=config['num_layers'], dropout=config['dropout'],
        output_range=output_range,
    ).to(device)

    criterion = AsymmetricRULLoss(alpha=config['alpha'])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config['lr'], weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-5
    )
    early_stopper = EarlyStopping(patience=config['patience'])

    train_ds = TimeSeriesDataset(data['X_train'], data['y_train'], data['rul_train'])
    val_ds = TimeSeriesDataset(data['X_val'], data['y_val'], data['rul_val'])
    test_ds = TimeSeriesDataset(data['X_test'], data['y_test'], data['rul_test'])

    train_loader = DataLoader(
        train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0
    )
    test_loader = DataLoader(
        test_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0
    )

    best_val_rmse = float('inf')
    pbar = tqdm(range(1, config['num_epochs'] + 1), desc=f'  {dataset_name}')
    for epoch in pbar:
        tl = train_rul_epoch(model, train_loader, optimizer, criterion,
                              device, config['gradient_clip'])
        vm = evaluate_rul(model, val_loader, device)
        scheduler.step()
        pbar.set_postfix({'loss': f'{tl:.3f}', 'RMSE': f"{vm['rmse']:.1f}"})

        if vm['rmse'] < best_val_rmse:
            best_val_rmse = vm['rmse']
            torch.save(model.state_dict(), f'{save_prefix}_{dataset_name}.pt')

        if early_stopper(vm['rmse']):
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(torch.load(
        f'{save_prefix}_{dataset_name}.pt', map_location=device
    ))
    test_metrics = evaluate_rul(model, test_loader, device)
    test_metrics['dataset'] = dataset_name
    test_metrics['n_train'] = len(data['X_train'])
    test_metrics['n_test'] = len(data['X_test'])
    test_metrics['n_features'] = n_feat
    test_metrics['n_sensors'] = len(sensor_cols)

    print(f"  Test RMSE: {test_metrics['rmse']:.2f} | MAE: {test_metrics['mae']:.2f}")
    return model, test_metrics


# ─ Run benchmark on both datasets ─
print("\n" + "="*60)
print("BENCHMARK MODE: Synthetic ESP vs NASA CMAPSS FD001")
print("="*60)

# 1. Synthetic
print("\n[1/2] Loading synthetic data...")
syn_df = generate_esp_dataset(n_wells=50, timesteps_per_well=4000, random_seed=42)
syn_df['failure'] = (syn_df['machine_status'] == 'BROKEN').astype(int)
syn_df[SYNTHETIC_SENSOR_COLS] = syn_df[SYNTHETIC_SENSOR_COLS].ffill().fillna(0)
syn_rul = _compute_rul(syn_df['failure'].values.astype(np.float32))
syn_rul = np.clip(syn_rul, 0, CONFIG['clip_rul']).astype(np.float32)
syn_rul[syn_rul == CONFIG['clip_rul']] = -1
X_sw, y_sw, r_sw = _sliding_window(
    syn_df[SYNTHETIC_SENSOR_COLS].values.astype(np.float32),
    syn_df['failure'].values.astype(np.float32), syn_rul,
    CONFIG['window_size'], CONFIG['step_size'],
)
syn_data = _split_and_scale(
    X_sw, y_sw, r_sw, SYNTHETIC_SENSOR_COLS,
    CONFIG['val_split'], CONFIG['test_split'], 42
)

# 2. CMAPSS
print("\n[2/2] Loading CMAPSS FD001...")
try:
    cmapss_data = load_cmapss(
        data_dir='../data/raw/cmapss', subset='FD001',
        window_size=CONFIG['window_size'], step_size=CONFIG['step_size'],
        clip_rul=130, val_split=CONFIG['val_split'],
    )
    cmapss_cols = cmapss_data['feature_names']
    print(f"  CMAPSS loaded: {cmapss_data['X_train'].shape}")
except Exception as e:
    print(f"  CMAPSS not found: {e}")
    print("  Run: !python scripts/download_data.py --dataset cmapss")
    cmapss_data = None
    cmapss_cols = []

# Train on both
results = OrderedDict()
print("\n" + "-"*60)
print("TRAINING")
print("-"*60)

_, syn_metrics = train_and_evaluate_on_dataset(
    'synthetic', syn_data, SYNTHETIC_SENSOR_COLS, CONFIG, DEVICE, 'checkpoints/bench'
)
results['Synthetic ESP'] = syn_metrics

if cmapss_data is not None:
    _, cmapss_metrics = train_and_evaluate_on_dataset(
        'cmapss', cmapss_data, cmapss_cols, CONFIG, DEVICE, 'checkpoints/bench'
    )
    results['CMAPSS FD001'] = cmapss_metrics

# ─ Comparison Table ─
print("\n" + "="*70)
print("BENCHMARK RESULTS")
print("="*70)
print(f"{'Metric':<20} {'Synthetic ESP':>15} {'CMAPSS FD001':>15}")
print("-"*70)

for key, label in [('rmse', 'RMSE'), ('mae', 'MAE'), ('nasa_score', 'NASA Score'), ('r2', 'R-squared')]:
    syn_val = results['Synthetic ESP'].get(key, float('nan'))
    cmapss_val = results.get('CMAPSS FD001', {}).get(key, float('nan'))
    print(f"{label:<20} {syn_val:>15.2f} {cmapss_val:>15.2f}")

print("-"*70)
n_train_s = results['Synthetic ESP'].get('n_train', 0)
n_train_c = results.get('CMAPSS FD001', {}).get('n_train', 0)
n_feat_s = results['Synthetic ESP'].get('n_features', 0)
n_feat_c = results.get('CMAPSS FD001', {}).get('n_features', 0)
print(f"{'Train windows':<20} {n_train_s:>15,} {n_train_c:>15,}")
print(f"{'Features':<20} {n_feat_s:>15} {n_feat_c:>15}")
print(f"{'Sensors':<20} {results['Synthetic ESP'].get('n_sensors', 0):>15} {results.get('CMAPSS FD001', {}).get('n_sensors', 0):>15}")
print("="*70)